In [20]:
import pandas as pd
import numpy as np
df1 = pd.read_csv("CRMLSListing202602.csv")
df2 = pd.read_csv("CRMLSListing202603.csv")
df = pd.concat([df1, df2], ignore_index=True)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print("=== SHAPE ===", df.shape)
print(df.head())


=== SHAPE === (71326, 81)
   OriginalListPrice  ListingKey CloseDate  ClosePrice ListAgentFirstName  \
0          1550000.0  1155725741       NaN         NaN              Filip   
1           789000.0  1155724256       NaN         NaN            Allison   
2           489000.0  1155721921       NaN         NaN            Lindsey   
3          1759000.0  1155718331       NaN         NaN              Terri   
4           510000.0  1155718210       NaN         NaN          Gabriella   

  ListAgentLastName   Latitude   Longitude         UnparsedAddress  \
0          Niculete  34.195867 -118.431627     6901 Woodman Avenue   
1            Gabski  39.753118 -121.640809  5772 Acorn Ridge Drive   
2              Harn  35.405254 -120.864331  3329 Panorama Drive 30   
3            Temple  33.143327 -117.320274       4747 Bryce Circle   
4          Madrigal  34.044249 -117.407775              18177 11th   

  PropertyType  LivingArea  ListPrice  DaysOnMarket  \
0         Land         NaN  1550000

In [21]:
print("\n=== INFO ===")
print(df.info())


=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 71326 entries, 0 to 71325
Data columns (total 81 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   OriginalListPrice             71077 non-null  float64
 1   ListingKey                    71326 non-null  int64  
 2   CloseDate                     9888 non-null   str    
 3   ClosePrice                    7733 non-null   float64
 4   ListAgentFirstName            70929 non-null  str    
 5   ListAgentLastName             71323 non-null  str    
 6   Latitude                      70229 non-null  float64
 7   Longitude                     70257 non-null  float64
 8   UnparsedAddress               71119 non-null  str    
 9   PropertyType                  71326 non-null  str    
 10  LivingArea                    62181 non-null  float64
 11  ListPrice                     71169 non-null  float64
 12  DaysOnMarket                  71326 non-null  int64  
 13

In [22]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing": missing,
    "Missing%": missing_pct
}).sort_values(by="Missing%", ascending=False)
print("\n=== MISSING ===")
print(missing_df.head(20))



=== MISSING ===
                              Missing  Missing%
AboveGradeFinishedArea          71326    100.00
FireplacesTotal                 71326    100.00
ElementarySchoolDistrict        71326    100.00
MiddleOrJuniorSchoolDistrict    71326    100.00
CoveredSpaces                   71326    100.00
TaxYear                         71247     99.89
BelowGradeFinishedArea          71005     99.55
BusinessType                    70840     99.32
TaxAnnualAmount                 70774     99.23
CoBuyerAgentFirstName           70637     99.03
BuilderName                     68901     96.60
LotSizeDimensions               67388     94.48
ElementarySchool                64892     90.98
MiddleOrJuniorSchool            64884     90.97
ClosePrice                      63593     89.16
BuyerOfficeAOR                  62977     88.29
HighSchool                      62824     88.08
BuyerOfficeName                 62201     87.21
BuyerOfficeName.1               62201     87.21
BuyerAgentMlsId        

I will remove columns with more than 80% missing values, as they provide limited information and may negatively impact the analysis. However, critical variables should be carefully evaluated before removal, even if they exhibit a high missing rate.

In [23]:
num_cols = df.select_dtypes(include=['number']).columns
cat_cols = df.select_dtypes(include=['object', 'string']).columns

print("\n=== NUM COLS ===", len(num_cols))
print("\n=== CAT COLS ===", len(cat_cols))


=== NUM COLS === 35

=== CAT COLS === 46


In [25]:
key_num_cols = [
    'ListPrice',
    'LivingArea',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'LotSizeSquareFeet',
    'YearBuilt'
]

key_cat_cols = [
    'City',
    'PropertyType',
    'PropertySubType',
    'MlsStatus',
    'ListingAgentName',
    'ListOfficeName'
]
print("\n=== NUMERIC SUMMARY ===")
print(df[key_num_cols].describe().round(2))


=== NUMERIC SUMMARY ===
          ListPrice  LivingArea  BedroomsTotal  BathroomsTotalInteger  \
count  7.116900e+04    62181.00       62397.00               65310.00   
mean   1.052575e+06     1942.12           3.17                   2.55   
std    2.187223e+06     1616.92           1.95                   1.95   
min    0.000000e+00        0.00           0.00                   0.00   
25%    1.550000e+05     1184.00           2.00                   2.00   
50%    6.650000e+05     1617.00           3.00                   2.00   
75%    1.200000e+06     2276.00           4.00                   3.00   
max    1.000000e+08    99999.00         149.00                 149.00   

       LotSizeSquareFeet  YearBuilt  
count       6.512500e+04   65415.00  
mean        4.400918e+05    1979.02  
std         1.698022e+07      28.44  
min         0.000000e+00    1776.00  
25%         5.293000e+03    1960.00  
50%         7.544000e+03    1980.00  
75%         1.800000e+04    2002.00  
max         2

In [26]:
for col in key_cat_cols:
    if col in df.columns:
        print(f"\n=== {col} ===")
        print(df[col].value_counts().head(10))


=== City ===
City
Los Angeles     6035
San Diego       2914
San Jose        1461
Irvine          1032
Long Beach       842
Oakland          793
Palm Springs     723
Palm Desert      722
Lancaster        652
Riverside        627
Name: count, dtype: int64

=== PropertyType ===
PropertyType
Residential            46016
ResidentialLease       13561
Land                    4721
ResidentialIncome       3059
ManufacturedInPark      2028
CommercialSale          1040
CommercialLease          693
BusinessOpportunity      208
Name: count, dtype: int64

=== PropertySubType ===
PropertySubType
SingleFamilyResidence    38959
Condominium              12709
Townhouse                 3793
Apartment                 1766
Duplex                    1444
ManufacturedOnLand         679
Triplex                    594
Quadruplex                 530
MixedUse                   463
Office                     352
Name: count, dtype: int64

=== MlsStatus ===
MlsStatus
Active                 47835
Pending          

Utilizing the `value_counts()` method helps identify key patterns within a dataset—for instance, which cities have the highest number of listings and which property types are most prevalent. For example, cities such as Los Angeles and San Diego boast the highest listing counts, while residential properties clearly dominate the market.